In [1]:
import os

# Disable proxy for all HTTP requests
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("no_proxy", None)
os.environ.pop("NO_PROXY", None)

from dataclasses import dataclass

from autogen_core import AgentId, MessageContext, RoutedAgent, SingleThreadedAgentRuntime, message_handler


@dataclass
class MyMessageType:
    content: str

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelFamily

# Name here is agent type (agent id = (agent type, agent key))
class MyAssistant(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(
            model="Qwen/Qwen3-4B-Thinking-2507",
            base_url="http://gpu14:8000/v1",
            model_info={
                "vision": False,
                "function_calling": False,
                "json_output": False,
                "family": ModelFamily.UNKNOWN,
                "structured_output": False,
            },
        )
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: MyMessageType, ctx: MessageContext) -> None:
        print(f"{self.id.type} received message: {message.content}")
        response = await self._delegate.on_messages(
            [TextMessage(content=message.content, source="user")], ctx.cancellation_token
        )
        print(f"{self.id.type} responded: {response.chat_message}")

In [3]:
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await MyAssistant.register(runtime, "my_assistant", lambda: MyAssistant("my_assistant"))

AgentType(type='my_assistant')

In [4]:
runtime.start()  # Start processing messages in the background.
await runtime.send_message(MyMessageType("Hello, World!"), AgentId("my_assistant", "default"))
await runtime.stop()  # Stop processing messages in the background.

my_assistant received message: Hello, World!
my_assistant responded: id='6bd932cb-f790-4bbf-b252-55349a2953ba' source='my_assistant' models_usage=RequestUsage(prompt_tokens=43, completion_tokens=4873) metadata={} created_at=datetime.datetime(2026, 2, 6, 6, 40, 55, 541713, tzinfo=datetime.timezone.utc) content='Okay, the user said "Hello, World!" which is a common greeting. I need to respond appropriately.\n\nFirst, I should acknowledge their message. Since it\'s a simple hello, the standard response is to greet them back. Maybe say "Hello! How can I assist you today?" That\'s friendly and opens the door for them to ask for help.\n\nWait, the instructions say to use tools if needed. But in this case, there\'s no tool required. It\'s a straightforward greeting. So I don\'t need to call any tools here.\n\nI should make sure not to add any extra information unless necessary. Just a polite response. Let me check if there\'s any hidden context, but the user\'s message is just "Hello, World!"